# Tema: Transformaciones PySpark

## Objetivos
Limpiar nulos, convertir tipos y deduplicar de forma determinista.

## Conceptos importantes para el examen
Funciones nativas; unionByName; ventanas; try_cast; inmutabilidad.

**Dificultad:** Básico · **Tiempo estimado:** 55 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_02_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

In [ ]:
dirty = employees.select("employee_id", "name", F.col("salary").cast("string").alias("salary_raw"))
dirty = dirty.unionByName(spark.createDataFrame([(2, "  ANA  ", "no_valido"), (19, None, "42000")], dirty.schema))
versions = employees.unionByName(employees.filter("employee_id = 2").withColumn("updated_at", F.to_timestamp(F.lit("2026-02-01"))))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Normalizar texto

In [ ]:
display(dirty.withColumn("name", F.initcap(F.trim("name"))))

### 2. Conversión tolerante

In [ ]:
typed = dirty.withColumn("salary_number", F.expr("try_cast(salary_raw AS INT)"))
display(typed.filter("salary_number IS NULL"))

### 3. Última versión

In [ ]:
w = Window.partitionBy("employee_id").orderBy(F.col("updated_at").desc())
display(versions.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Rellena solo los nombres nulos con 'Sin nombre'.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Divide dirty en válidos y cuarentena según la conversión del salario.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Crea bandas junior (<40.000), mid (<50.000) y senior.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Deduplica versions eligiendo la fecha más reciente y comprueba que quedan 18 filas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Divide name por espacios, renombra salary a annual_salary y elimina created_at.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** fillna acepta un diccionario.

**Pista 2:** Prueba isNull/isNotNull.

**Pista 3:** when y otherwise.

**Pista 4:** row_number; dropDuplicates no define qué versión gana.

**Pista 5:** split, withColumnRenamed y drop.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
display(dirty.fillna({"name": "Sin nombre"}))

### Solución 2

In [ ]:
typed = dirty.withColumn("salary_number", F.expr("try_cast(salary_raw AS INT)"))
valid = typed.filter("salary_number IS NOT NULL")
quarantine = typed.filter("salary_number IS NULL")
assert valid.count() + quarantine.count() == dirty.count()
display(quarantine)

### Solución 3

In [ ]:
display(employees.withColumn("band", F.when(F.col("salary") < 40000, "junior").when(F.col("salary") < 50000, "mid").otherwise("senior")))

### Solución 4

In [ ]:
w = Window.partitionBy("employee_id").orderBy(F.col("updated_at").desc())
latest = versions.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
assert latest.count() == 18
assert latest.filter("employee_id = 2").first().updated_at == datetime(2026, 2, 1)
# Con empates reales se necesita otra columna de secuencia única.

### Solución 5

In [ ]:
display(employees.withColumn("name_parts", F.split("name", " ")).withColumnRenamed("salary", "annual_salary").drop("created_at"))

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Cómo eliges la versión más reciente de cada clave?

A. dropDuplicates global

B. row_number con orden por secuencia

C. limit global

D. union

### Pregunta 2
¿Qué devuelve try_cast de 'abc' a INT?

A. 0

B. Borra la fila

C. NULL

D. 1

### Pregunta 3
¿Qué combina DataFrames por nombre de columna?

A. unionByName

B. crossJoin

C. orderBy

D. coalesce

### Respuestas y explicación
**1. B** — La ventana explicita qué fila gana.

**2. C** — Puedes detectar el nulo y enviar el registro a cuarentena.

**3. A** — Evita depender de la posición.

## PARTE 6 - RETO FINAL
Escribe valid_employees y quarantine_employees como Delta. Conserva el valor original y el motivo del rechazo; demuestra conservación de filas.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
